# BFCL Inference on Kaggle T4 — Fixed

Key fixes vs. the original notebook:
1. **Install vLLM explicitly** — Kaggle's base image doesn't include it, and `pip install -e . --no-deps` skips it.
2. **Use `XFORMERS` attention backend** — T4 (compute 7.5, Turing) doesn't support `TRITON_ATTN` or FlashAttention-2/FlashInfer.
3. **Force `float16`** — T4 has no bfloat16 support; Qwen2.5 defaults to bf16 and will crash without `--dtype float16`.
4. **Memory limits for 16 GB VRAM** — cap `gpu-memory-utilization` and `max-model-len` so vLLM fits.
5. Drop the unneeded `vllm serve → api_server` patch — once vLLM is installed properly, the CLI works.

In [ ]:
# Step 1: Clone the repo
!git clone https://github.com/Tusherbhomik/gorilla.git /kaggle/working/gorilla

In [ ]:
# Step 2: Navigate to BFCL directory
%cd /kaggle/working/gorilla/berkeley-function-call-leaderboard

In [ ]:
# Step 3: Install vLLM + BFCL deps
# vLLM 0.6.x is the last lineage that reliably works on T4 (Turing, SM 7.5).
# Newer vLLM (0.7+) drops Turing support in many kernels.

# Install vllm + a compatible torch first
!pip install -q vllm==0.6.3.post1

# BFCL deps
!pip install -q typer tabulate overrides python-dotenv tenacity tqdm filelock networkx

# Install bfcl_eval itself (no-deps to avoid downgrading vllm/torch)
!pip install -q -e . --no-deps

# Remove flashinfer if it sneaked in — T4 can't use it
!pip uninstall -y flashinfer flashinfer-python 2>/dev/null || true

# Sanity check
!python -c "import vllm; print('vllm', vllm.__version__)"
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'cap', torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None)"

In [ ]:
# Step 4: Patch base_oss_handler.py with T4-friendly vLLM args
# We add: --dtype float16, --gpu-memory-utilization 0.90, --max-model-len 4096,
#         --enforce-eager (skips torch.compile, avoids Turing issues)
# Keep the `vllm serve ...` form — it works once vllm is installed.

import re

handler_path = '/kaggle/working/gorilla/berkeley-function-call-leaderboard/bfcl_eval/model_handler/local_inference/base_oss_handler.py'

with open(handler_path, 'r') as f:
    content = f.read()

# Find the model path arg and append T4-specific flags right after it.
# Pattern: "vllm", "serve", str(self.model_path_or_id),
marker = 'str(self.model_path_or_id),'
extra_args = (
    'str(self.model_path_or_id),\n'
    '                            "--dtype", "float16",\n'
    '                            "--gpu-memory-utilization", "0.90",\n'
    '                            "--max-model-len", "4096",\n'
    '                            "--enforce-eager",'
)

if '--dtype' not in content:
    # Only patch the first occurrence (the vllm serve invocation)
    content = content.replace(marker, extra_args, 1)
    with open(handler_path, 'w') as f:
        f.write(content)
    print('Handler patched with T4 flags.')
else:
    print('Already patched, skipping.')

# Show the relevant block to verify
!grep -n -A 12 '"vllm"' {handler_path} | head -30

In [ ]:
# Step 5: Generate (inference)
# XFORMERS = the only attention backend that works on Turing (T4).
# TRITON_ATTN, FLASHINFER, FLASH_ATTN all require Ampere (SM 8.0+).

!VLLM_ATTENTION_BACKEND=XFORMERS \
 VLLM_WORKER_MULTIPROC_METHOD=spawn \
 python -m bfcl_eval generate \
    --model tusherbhomik/qwen2.5-1.5b-hgr-5340-r2 \
    --test-category simple_python \
    --num-gpus 1 \
    --backend vllm \
    --allow-overwrite

In [ ]:
# Step 6: Evaluate
!python -m bfcl_eval evaluate \
    --model tusherbhomik/qwen2.5-1.5b-hgr-5340-r2 \
    --test-category simple_python